---
title: "Data Collection"
format:
  html:
    embed-resources: true
    code-fold: true
    toc: true
---

## Overview

This tab contains all the information related to the process of data collection, initial filtering & cleaning, and some initial exploratory data analysis (EDA).

## Introduction

The data provided for this project contains all the posts of Reddit from January 2022 to March 2023. It has been hosted in Azure blobs by our DSAN 6000 professors at Georgetown University. The data is divided into 2 files, submissions and comments. While they follow a similar structure, they have some different attributes and names. The submissions file refers to the posts made by users, while the comments file refers to the comments made by users on the posts. Thus, the comments file is much larger than the submissions file. Underneath you can see the first 5 rows of each file after selecting the entries that we will be using on this project.

#### A note on terminology

You may see us refer to submissions and comments directly, or more generally to "posts" - over the course of the project, we settled on using the term "post" to refer to any single row of textual data used in our pipeline, which is inclusive of both submissions & comments. 

## Cleaning Process Summary

The cleaning was submitted as a job in Azure. The actual file can be found under the folder `spark_jobs/Clean_All_Data/` in the file `Clean_job.py`. The cleaning process was done in the following steps:

1. Loaded the data from the Azure blob.

2. Removed subreddits that would skew the perspective of the data.
   - democrats
   - Republican
   - The_Donald
   - EnoughTrumpSpam
   - Fuckthealtright
   - Communism
   - FullCommunism
   - Anarchism
   - AntiWork
   - GenZedong
   - Conspiracy
   - DarkEnlightenment
   - NeoReaction
   - Identitarian
   - Fascist
   - Socialism
   - Conservative
   - Patriot
   - ConsumeProduct

3. Removed posts that were not from January 2022 until November 2022.

4. Removed all the posts that were empty, [removed], or [deleted].

5. Then, we filtered to only select the posts that were politics related. To do so wey used certain keywords to filter the data. This keywords can be found in the folder `data/keywords/`.

6. Due to our limitations, we could only process english posts, so we filtered the data to not include any other language.


**Below is a breakdown of this process:**

## Code & Process

### Getting the Data

**We used a custom environment that contained NLTK and Langdetect packages which were then used to clean and filter our Reddit data**  

First, we imported necessary packages and initialized the spark session.

In [ ]:
#initialize the session
spark

# Importing necessary libraries for English language detect and text lematization
import langdetect
from langdetect import detect
from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
import nltk

**In order to collect the data, we connected to the Azure Blob where the Reddit data is housed and grabbed the Comments and Submissions datasets**

In [2]:
blob_account_name = "dsan6000fall2024"
blob_container_name = "reddit-project"
wasbs_base_url = (
    f"wasbs://{blob_container_name}@{blob_account_name}.blob.core.windows.net/"
)

# defining paths for comments and submissions
comments_path = f"{wasbs_base_url}202101-202303/comments/"
submissions_path = f"{wasbs_base_url}202101-202303/submissions/"

# printing to verify paths
print("Comments Path:", comments_path)
print("Submissions Path:", submissions_path)

StatementMeta(16fd2129-cd11-4016-9021-0a19ae621014, 83, 7, Finished, Available, Finished)

Comments Path: wasbs://reddit-project@dsan6000fall2024.blob.core.windows.net/202101-202303/comments/
Submissions Path: wasbs://reddit-project@dsan6000fall2024.blob.core.windows.net/202101-202303/submissions/


In [3]:
# reading the parquet files
comments_df = spark.read.parquet(comments_path)
submissions_df = spark.read.parquet(submissions_path)

StatementMeta(16fd2129-cd11-4016-9021-0a19ae621014, 83, 8, Finished, Available, Finished)

**Once we had successfully read the data sets, we went ahead and performed simple EDA to get an idea of how many data points we were working with, as well as the data schemas**

In [6]:
# counting number of rows for each
comments_count = comments_df.count()
submissions_count = submissions_df.count()

print(f"Number of comments: {comments_count}")
print(f"Number of submissions: {submissions_count}")

StatementMeta(16fd2129-cd11-4016-9021-0a19ae621014, 5, 11, Finished, Available, Finished)

Number of comments: 6114480450
Number of submissions: 892160821


In [5]:
# printing Comments schema
print("Comments Schema:")
comments_df.printSchema()

StatementMeta(11387b7c-91c2-4dd1-a4f8-a460b59b7bdd, 14, 10, Finished, Available, Finished)

Comments Schema:
root
 |-- author: string (nullable = true)
 |-- author_cakeday: boolean (nullable = true)
 |-- author_flair_css_class: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- body: string (nullable = true)
 |-- can_gild: boolean (nullable = true)
 |-- controversiality: long (nullable = true)
 |-- created_utc: timestamp (nullable = true)
 |-- distinguished: string (nullable = true)
 |-- edited: string (nullable = true)
 |-- gilded: long (nullable = true)
 |-- id: string (nullable = true)
 |-- is_submitter: boolean (nullable = true)
 |-- link_id: string (nullable = true)
 |-- parent_id: string (nullable = true)
 |-- permalink: string (nullable = true)
 |-- retrieved_on: timestamp (nullable = true)
 |-- score: long (nullable = true)
 |-- stickied: boolean (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- subreddit_id: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [6]:
# printing Submissions schema
print("Submissions Schema:")
submissions_df.printSchema()

StatementMeta(11387b7c-91c2-4dd1-a4f8-a460b59b7bdd, 14, 11, Finished, Available, Finished)

Submissions Schema:
root
 |-- adserver_click_url: string (nullable = true)
 |-- adserver_imp_pixel: string (nullable = true)
 |-- archived: boolean (nullable = true)
 |-- author: string (nullable = true)
 |-- author_cakeday: boolean (nullable = true)
 |-- author_flair_css_class: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- author_id: string (nullable = true)
 |-- brand_safe: boolean (nullable = true)
 |-- contest_mode: boolean (nullable = true)
 |-- created_utc: timestamp (nullable = true)
 |-- crosspost_parent: string (nullable = true)
 |-- crosspost_parent_list: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- approved_at_utc: string (nullable = true)
 |    |    |-- approved_by: string (nullable = true)
 |    |    |-- archived: boolean (nullable = true)
 |    |    |-- author: string (nullable = true)
 |    |    |-- author_flair_css_class: string (nullable = true)
 |    |    |-- author_flair_text: string (nullable

#### A Disclaimer

**Please note that because there are too many data points to work with on this notebook, we will be sampling the datasets to 100 observations each.**

In [6]:
# Sample the datasets to include only 100 rows each
comments_sampled_df = comments_df.limit(100)
submissions_sampled_df = submissions_df.limit(100)

StatementMeta(16fd2129-cd11-4016-9021-0a19ae621014, 83, 11, Finished, Available, Finished)

In [7]:
# cache'ing the sampled DFs
comments_sampled_df.cache()
submissions_sampled_df.cache()

StatementMeta(16fd2129-cd11-4016-9021-0a19ae621014, 83, 12, Finished, Available, Finished)

DataFrame[adserver_click_url: string, adserver_imp_pixel: string, archived: boolean, author: string, author_cakeday: boolean, author_flair_css_class: string, author_flair_text: string, author_id: string, brand_safe: boolean, contest_mode: boolean, created_utc: timestamp, crosspost_parent: string, crosspost_parent_list: array<struct<approved_at_utc:string,approved_by:string,archived:boolean,author:string,author_flair_css_class:string,author_flair_text:string,banned_at_utc:string,banned_by:string,brand_safe:boolean,can_gild:boolean,can_mod_post:boolean,clicked:boolean,contest_mode:boolean,created:double,created_utc:double,distinguished:string,domain:string,downs:bigint,edited:boolean,gilded:bigint,hidden:boolean,hide_score:boolean,id:string,is_crosspostable:boolean,is_reddit_media_domain:boolean,is_self:boolean,is_video:boolean,likes:string,link_flair_css_class:string,link_flair_text:string,locked:boolean,media:string,mod_reports:array<string>,name:string,num_comments:bigint,num_crosspos

**Now we took a peek at what the datasets look like:**

In [8]:
# displaying each
print("Sampled Comments DataFrame:")
comments_sampled_df.show(1, truncate=False)

StatementMeta(16fd2129-cd11-4016-9021-0a19ae621014, 83, 13, Finished, Available, Finished)

Sampled Comments DataFrame:
+----------+--------------+----------------------+-----------------+------------------+--------+----------------+-------------------+-------------+------+------+-------+------------+---------+----------+--------------------------------------------------------+------------+-----+--------+---------------------+------------+----+-----+
|author    |author_cakeday|author_flair_css_class|author_flair_text|body              |can_gild|controversiality|created_utc        |distinguished|edited|gilded|id     |is_submitter|link_id  |parent_id |permalink                                               |retrieved_on|score|stickied|subreddit            |subreddit_id|year|month|
+----------+--------------+----------------------+-----------------+------------------+--------+----------------+-------------------+-------------+------+------+-------+------------+---------+----------+--------------------------------------------------------+------------+-----+--------+--------------

In [9]:
print("Sampled Submissions DataFrame:")
submissions_sampled_df.show(1, truncate=False)

StatementMeta(16fd2129-cd11-4016-9021-0a19ae621014, 83, 14, Finished, Available, Finished)

Sampled Submissions DataFrame:
+------------------+------------------+--------+------------+--------------+----------------------+-----------------+---------+----------+------------+-------------------+----------------+---------------------+----------------+-------------+------+---------------+------+----------+---------+------+------+----------+--------+------+---------+----------------+----------------------+-------+--------+--------------------+---------------+------+-----+------------------------+-------------+------------+--------------+-------------+-------+-----------------------+-----------------------------------------------------------------------------------+------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Sub-Setting the Data

**The following cells created new functions that would 1) detect English language only and 2) load and lemmatized keywords**

In [9]:
# First, we fix the pathing so that we can successfully locate our repo in AzureML, as well as the full paths for our code
# we will be using Jude's path.
import os

repo_path = os.path.join(os.getcwd(), "Users", "jjm385", "fall-2024-project-team-11")
os.chdir(repo_path)

# lets verify the updated working directory
print("Updated Working Directory:", os.getcwd())

StatementMeta(11387b7c-91c2-4dd1-a4f8-a460b59b7bdd, 14, 14, Finished, Available, Finished)

Updated Working Directory: /synfs/notebook/14/aml_notebook_mount/Users/jjm385/fall-2024-project-team-11


**The following function was created using the LangDetect package, which we used to only select posts written in English.**

In [10]:
# defining a UDF to detect English text. This will be used for both the Comments and Submissions Datasets
def is_english(text):
    try:
        return detect(text) == "en"
    except:
        return False

is_english_udf = udf(is_english, BooleanType())

StatementMeta(11387b7c-91c2-4dd1-a4f8-a460b59b7bdd, 14, 15, Finished, Available, Finished)

**The following cell created a function that used NLTK and our list of keywords to check for our keywords (both original and lemmatized) within each post.**

In [11]:
# Download NLTK data if needed
nltk.download('wordnet')
nltk.download('omw-1.4')

# Initialize WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

# Function to load and lemmatize keywords
def load_and_lemmatize_keywords(file_path):
    with open(file_path, 'r') as f:
        keywords = [line.strip() for line in f.readlines()]
    # Lemmatize each keyword
    lemmatized_keywords = [lemmatizer.lemmatize(word) for word in keywords]
    return lemmatized_keywords

# Load and lemmatize the `_Lemmatized.txt` files
keywords_left_lemmatized = load_and_lemmatize_keywords("website-source/keywords/Keywords_Left_Lemmatized.txt")
keywords_right_lemmatized = load_and_lemmatize_keywords("website-source/keywords/Keywords_Right_Lemmatized.txt")
keywords_non_partisan_lemmatized = load_and_lemmatize_keywords("website-source/keywords/Keywords_Non_Partisan_Lemmatized.txt")

# Load the unchanged keywords (as in the previous code)
def load_keywords(file_path):
    with open(file_path, 'r') as f:
        return [line.strip() for line in f.readlines()]

keywords_left_unchanged = load_keywords("website-source/keywords/Keywords_Left_Unchanged.txt")
keywords_right_unchanged = load_keywords("website-source/keywords/Keywords_Right_Unchanged.txt")
keywords_non_partisan_unchanged = load_keywords("website-source/keywords/Keywords_Non_Partisan_Unchanged.txt")

# Combine all keywords into a single list
all_keywords = set(
    keywords_left_unchanged + keywords_right_unchanged + keywords_non_partisan_unchanged +
    keywords_left_lemmatized + keywords_right_lemmatized + keywords_non_partisan_lemmatized
)

# Creating function to check for keywords in each post and comment
def contains_keyword(body):
    if body is None:
        return False
    for keyword in all_keywords:
        if keyword in body:
            return True
    return False

is_keyword_present = udf(contains_keyword, BooleanType())

StatementMeta(11387b7c-91c2-4dd1-a4f8-a460b59b7bdd, 14, 16, Finished, Available, Finished)

[nltk_data] Downloading package wordnet to /home/trusted-service-
[nltk_data]     user/nltk_data...
[nltk_data] Downloading package omw-1.4 to /home/trusted-service-
[nltk_data]     user/nltk_data...


**Now that we had both of our functions ready to go, we then moved on to sub-setting our data for both Comments and Submissions datasets** 

### Comments Subsetting

In [12]:
# selecting only necessary columns 
comments_filtered = comments_sampled_df.select(
    "id",          # unique identifier
    "author",      # author 
    "body",        # text content
    "created_utc", # timestamp
    "year",        
    "month",       
    "subreddit"    
)

# filter out the subreddits that are affiliated with the extreme political viewpoints
comments_df = comments_filtered.filter(
    (~comments_filtered["subreddit"].isin(
        "democrats", "Republican", "The_Donald", "EnoughTrumpSpam", "Fuckthealtright",
        "Communism", "FullCommunism", "Anarchism", "AntiWork", "GenZedong", "Conspiracy",
        "DarkEnlightenment", "NeoReaction", "Identitarian", "Fascist", "Socialism", 
        "Conservative", "Patriot", "ConsumeProduct")) &
    (comments_filtered["year"] == 2022) &
    (comments_filtered["month"].between(1, 11)) &
    (comments_filtered["body"] != "[removed]") &  # Exclude removed comments
    (comments_filtered["body"] != "[deleted]") &  # Exclude deleted comments
    (comments_filtered["body"] != "") &           # Exclude empty comments
    (is_english_udf(comments_filtered["body"])) & # Ensure comments are in English
    (is_keyword_present(comments_filtered["body"]))  # Ensure keywords are present
)
comments_df.show(5)

StatementMeta(11387b7c-91c2-4dd1-a4f8-a460b59b7bdd, 14, 17, Finished, Available, Finished)

+---+------+----+-----------+----+-----+---------+
| id|author|body|created_utc|year|month|subreddit|
+---+------+----+-----------+----+-----+---------+
+---+------+----+-----------+----+-----+---------+



### Submissions Subsetting

In [14]:
#selecting only necessary columns

submissions_filtered = submissions_sampled_df.select(
    "id",          # unique identifier
    "author",      # author 
    "title",       # post title
    "created_utc", # timestamp
    "num_comments",# could be useful for engagement
    "selftext",    # text of the post
    "year",        
    "month",       
    "subreddit"    
)

# filter out the subreddits that are affiliated with the extreme political viewpoints
submissions_df = submissions_filtered.filter(
    (~submissions_filtered["subreddit"].isin("democrats", "Republican", "The_Donald", 
    "EnoughTrumpSpam", "Fuckthealtright","Communism", "FullCommunism", "Anarchism", "AntiWork", "GenZedong",
    "Conspiracy", "DarkEnlightenment", "NeoReaction", "Identitarian", "Fascist", "Socialism", "Conservative", "Patriot", "ConsumeProduct"))  & 
    (submissions_filtered["year"] == 2022) & 
    (submissions_filtered["month"].between(1, 11)) &
    (submissions_filtered["selftext"].isNotNull()) & #grabs the non-link posts 
    (submissions_filtered["selftext"] != "[removed]") & #strips posts that were removed and have no text
    (submissions_filtered["selftext"] != "[deleted]") & #similar but with self deletion
    (submissions_filtered["selftext"] != "") & #gets rid of posts with no content
    is_english_udf(submissions_filtered["selftext"]) & # filters for English text
    is_keyword_present(submissions_filtered["selftext"])  # filters for keywords
)
submissions_df.show(5)

StatementMeta(11387b7c-91c2-4dd1-a4f8-a460b59b7bdd, 14, 19, Finished, Available, Finished)

+---+------+-----+-----------+------------+--------+----+-----+---------+
| id|author|title|created_utc|num_comments|selftext|year|month|subreddit|
+---+------+-----+-----------+------------+--------+----+-----+---------+
+---+------+-----+-----------+------------+--------+----+-----+---------+



**Please note that the sub-setting of the data successfully works, however no data is being displayed above since we are only using 100 data points on this notebook, and in this limited sample there are none of our goal posts present.**

**The code was ran successfully as a job against the entire Comments and Submissions datasets. The resulting data sets were then used to transform our data in order to make our labeling process easier.**  